In [1]:
import pandas as pd
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import json
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F

/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.9.1 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0729 11:28:22.000000 77829 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
language_classifier = pipeline(
    "text-classification",
    model = "papluca/xlm-roberta-base-language-detection"
)

def lang_result(text):
    results = language_classifier(
        text,
        truncation=True
    )
    return results[0]["label"]

theme_classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-base-zeroshot-v2.0",
    multi_label = True
)

def theme_result(text, theme_labels):
    return theme_classifier(
        text,
        candidate_labels=theme_labels,
        hypothesis_template="This post discusses {}.",
        multi_label=True
    )

model_name = "yangheng/deberta-v3-base-absa-v1.1"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()


def aspect_sentiment(text, aspect, batch_size=16, max_length=512, stride=64):
    encoded = tokenizer(
        text,
        aspect,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        padding=True,
        return_tensors="pt"
    )

    input_keys = ["input_ids", "attention_mask", "token_type_ids"]
    input_keys = [k for k in input_keys if k in encoded]

    all_probs = []

    with torch.inference_mode():
        n_chunks = encoded["input_ids"].shape[0]

        for start in range(0, n_chunks, batch_size):
            end = start + batch_size

            batch = {
                k: encoded[k][start:end].to(device)
                for k in input_keys
            }

            outputs = model(**batch)
            probs = F.softmax(outputs.logits, dim=-1)
            all_probs.append(probs)

    avg_probs = torch.cat(all_probs, dim=0).mean(dim=0).cpu()

    return {
        model.config.id2label[i]: float(avg_probs[i])
        for i in range(len(avg_probs))
    }

Device set to use mps:0
Device set to use mps:0


Using device: mps


/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [3]:
def remove_links(text):
    url_pattern = re.compile(r'\[https?://\S+\]|\(https?://\S+\)|\[www\.\S+\]|\(www\.\S+\)')
    cleaned_text = url_pattern.sub('', text)
    return cleaned_text

In [4]:
BASE_DIR = Path.cwd()
file_path = (
    BASE_DIR
    / "brightdata_social_exports"
    / "quora_datacenters_posts.json"
)
with file_path.open("r", encoding="utf-8") as f:
    quora_posts = json.load(f)

In [5]:
english_post_ids = []
a = 0

for q_post in quora_posts:
    unclean_text = q_post["post_text"]
    text = remove_links(unclean_text)
    language = lang_result(text)
    pid = q_post["post_id"]
    if language == 'en':
        english_post_ids.append(pid)
    a += 1
    print("Posts scanned:", a, end="\r")

KeyboardInterrupt: 

In [6]:
len(english_post_ids)

16

In [7]:
all_post_ids = []

env_post_ids = []
env_post_sentiments = []
env_post_sentiment_degrees = []

infr_post_ids = []
infr_post_sentiments = []
infr_post_sentiment_degrees = []

housing_post_ids = []
housing_post_sentiments = []
housing_post_sentiment_degrees = []

econ_post_ids = []
econ_post_sentiments = []
econ_post_sentiment_degrees = []

life_qual_post_ids = []
life_qual_post_sentiments = []
life_qual_post_sentiment_degrees = []

aesth_post_ids = []
aesth_post_sentiments = []
aesth_post_sentiment_degrees = []

gov_post_ids = []
gov_post_sentiments = []
gov_post_sentiment_degrees = []

tech_post_ids = []
tech_post_sentiments = []
tech_post_sentiment_degrees = []

not_useful_post_ids = []

themes = ["visual impact of datacenters", "infrastructure and utilities", "housing costs and property values", "economy and jobs", "quality of life, noise, and light pollution", "environmental impact", "government decisions and policies", "technology performance and growth"]

matched_posts = 0
more_than_one_theme_posts = 0

a = 0

for q_post in quora_posts:
    a += 1

    post_id = q_post["post_id"]
    if post_id not in english_post_ids:
        continue
    all_post_ids.append(post_id)
    unclean_text = q_post["post_text"]
    text = remove_links(unclean_text)

    final_labels = []

    theme_scores = theme_result(
        text,
        themes
    )
    
    for i in range(len(theme_scores['labels'])):
        if theme_scores['scores'][i] > 0.5:
            final_labels.append(theme_scores['labels'][i])
    
    if len(final_labels) > 0:
        matched_posts += 1
    else:
        not_useful_post_ids.append(post_id)
    
    if len(final_labels) > 1:
        more_than_one_theme_posts += 1
    

    for label in final_labels:
        theme_sentiment = aspect_sentiment(text, label)
        post_sentiment = max(theme_sentiment, key=theme_sentiment.get)
        post_degree = max(theme_sentiment.values())

        if label == "visual impact of datacenters":
            aesth_post_ids.append(post_id)
            aesth_post_sentiments.append(post_sentiment)
            aesth_post_sentiment_degrees.append(post_degree)
        if label == "infrastructure and utilities":
            infr_post_ids.append(post_id)
            infr_post_sentiments.append(post_sentiment)
            infr_post_sentiment_degrees.append(post_degree)
        if label == "housing costs and property values":
            housing_post_ids.append(post_id)
            housing_post_sentiments.append(post_sentiment)
            housing_post_sentiment_degrees.append(post_degree)
        if label == "economy and jobs":
            econ_post_ids.append(post_id)
            econ_post_sentiments.append(post_sentiment)
            econ_post_sentiment_degrees.append(post_degree)
        if label == "quality of life, noise, and light pollution":
            life_qual_post_ids.append(post_id)
            life_qual_post_sentiments.append(post_sentiment)
            life_qual_post_sentiment_degrees.append(post_degree)
        if label == "environmental impact":
            env_post_ids.append(post_id)
            env_post_sentiments.append(post_sentiment)
            env_post_sentiment_degrees.append(post_degree)
        if label == "government decisions and policies":
            gov_post_ids.append(post_id)
            gov_post_sentiments.append(post_sentiment)
            gov_post_sentiment_degrees.append(post_degree)
        if label == "technology performance and growth":
            tech_post_ids.append(post_id)
            tech_post_sentiments.append(post_sentiment)
            tech_post_sentiment_degrees.append(post_degree)

        print("Posts scanned:", a, end="\r")

print("Total posts scanned:", len(all_post_ids))
print("Total posts with a theme:", matched_posts)
print("Found environmental posts:", len(env_post_ids))
print("Found infrastructure posts:", len(infr_post_ids))
print("Found housing posts:", len(housing_post_ids))
print("Found economic posts:", len(econ_post_ids))
print("Found life quality posts:", len(life_qual_post_ids))
print("Found aesthetic posts:", len(aesth_post_ids))
print("Found government posts:", len(gov_post_ids))
print("Found technological posts:", len(tech_post_ids))
print(not_useful_post_ids)

print(gov_post_sentiments)
print(gov_post_sentiment_degrees)

Total posts scanned: 16
Total posts with a theme: 9
Found environmental posts: 0
Found infrastructure posts: 3
Found housing posts: 0
Found economic posts: 1
Found life quality posts: 0
Found aesthetic posts: 0
Found government posts: 0
Found technological posts: 8
['QW5zd2VyQDA6MTQ3Nzc0Mzc1MTQ3OTk4Mg==', 'QW5zd2VyQDA6MzQ2MzQ4MDUy', 'QW5zd2VyQDA6NDYxMTM0MDY=', 'QW5zd2VyQDA6MjY5NDM3NjQ0', 'QW5zd2VyQDA6MzgyMzcwNzkw', 'QW5zd2VyQDA6ODcxMjI2MTg=', 'QW5zd2VyQDA6MTQ3Nzc0Mzc3NjgyODE1NQ==']
[]
[]


In [8]:
posts_by_id = {post["post_id"]: post for post in quora_posts}
life_qual_links = []

for id in life_qual_post_ids:
    post = posts_by_id.get(id)
    life_qual_links.append(post["url"])

print(life_qual_links)

[]


In [9]:
theme_lists = [env_post_ids, infr_post_ids, housing_post_ids, econ_post_ids, life_qual_post_ids, aesth_post_ids, gov_post_ids, tech_post_ids]
posts = pd.DataFrame(columns=["ids", "text", "date", "upvotes", "number of answers", "shares", "views", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology", "AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"])
datacenters_keywords = ["datacenter", "data center", "datacentre", "data centre"]

posts_by_id = {post["post_id"]: post for post in quora_posts}

for theme in theme_lists:
    df1 = pd.DataFrame(columns=["ids", "text", "date", "upvotes", "number of answers", "shares", "views", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology", "AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"])
    post_ids = []
    post_texts = []
    post_dates = []
    post_upvotes = []
    post_num_answers = []
    post_shares = []
    post_views = []


    for pid in theme:
        post_ids.append(pid)
        post = posts_by_id.get(pid)
        post_texts.append(post["post_text"])
        post_dates.append(post["post_date"])
        post_upvotes.append(post["upvotes"])
        post_num_answers.append(post["over_all_answers"])
        post_shares.append(post["shares"])
        post_views.append(post["views"])


    df1["ids"] = post_ids
    df1["text"] = post_texts
    df1["date"] = post_dates
    df1["upvotes"] = post_upvotes
    df1["number of answers"] = post_num_answers
    df1["shares"] = post_shares
    df1["views"] = post_views

    
    for col in ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology"]:
        df1[col] = False

    for col in ["environment sentiment", "environment sentiment degree", "infrastructure sentiment", "infrastructure sentiment degree", "housing sentiment", "housing sentiment degree", "economy sentiment", "economy sentiment degree", "life quality sentiment", "life quality sentiment degree", "aesthetics sentiment", "aesthetics sentiment degree", "government sentiment", "government sentiment degree", "technology sentiment", "technology sentiment degree"]:
        df1[col] = None

    if theme == env_post_ids:
        df1["environment"] = True
        df1["environment sentiment"] = env_post_sentiments
        df1["environment sentiment degree"] = env_post_sentiment_degrees
    if theme == infr_post_ids:
        df1["infrastructure"] = True
        df1["infrastructure sentiment"] = infr_post_sentiments
        df1["infrastructure sentiment degree"] = infr_post_sentiment_degrees
    if theme == housing_post_ids:
        df1["housing"] = True
        df1["housing sentiment"] = housing_post_sentiments
        df1["housing sentiment degree"] = housing_post_sentiment_degrees
    if theme == econ_post_ids:
        df1["economy"] = True
        df1["economy sentiment"] = econ_post_sentiments
        df1["economy sentiment degree"] = econ_post_sentiment_degrees
    if theme == life_qual_post_ids:
        df1["life quality"] = True
        df1["life quality sentiment"] = life_qual_post_sentiments
        df1["life quality sentiment degree"] = life_qual_post_sentiment_degrees
    if theme == aesth_post_ids:
        df1["aesthetics"] = True
        df1["aesthetics sentiment"] = aesth_post_sentiments
        df1["aesthetics sentiment degree"] = aesth_post_sentiment_degrees
    if theme == gov_post_ids:
        df1["government"] = True
        df1["government sentiment"] = gov_post_sentiments
        df1["government sentiment degree"] = gov_post_sentiment_degrees
    if theme == tech_post_ids:
        df1["technology"] = True
        df1["technology sentiment"] = tech_post_sentiments
        df1["technology sentiment degree"] = tech_post_sentiment_degrees
    posts = pd.concat([posts, df1], ignore_index=True)

/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_77829/1995708874.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  posts = pd.concat([posts, df1], ignore_index=True)
/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_77829/1995708874.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  posts = pd.concat([posts, df1], ignore_index=True)
/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_77829/1995708874.py:76: FutureWarning: The behavior of DataFrame concatenation wi

In [28]:
len(posts)

83

In [29]:
posts = posts.astype({
    "ids": "string",
    "text": "string",
    "date": "string",
    "upvotes": "int64",
    "number of answers": "int64",
    "shares": "int64",
    "views": "int64"
})

In [30]:
grouping_cols = ["ids", "text", "date", "upvotes", "number of answers", "shares", "views"]

theme_cols = ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology"]
sent_cols  = ["environment sentiment", "environment sentiment degree", "infrastructure sentiment", "infrastructure sentiment degree", "housing sentiment", "housing sentiment degree", "economy sentiment", "economy sentiment degree", "life quality sentiment", "life quality sentiment degree", "aesthetics sentiment", "aesthetics sentiment degree", "government sentiment", "government sentiment degree", "technology sentiment", "technology sentiment degree"]

def first_non_null(s):
    return s.dropna().iloc[0] if s.notna().any() else np.nan

agg = {c: "max" for c in theme_cols}              # True if any True
agg.update({c: first_non_null for c in sent_cols}) # keep the real sentiment if present

posts = posts.groupby(grouping_cols, as_index=False, dropna=False).agg(agg)

In [31]:
print(posts.columns)

Index(['ids', 'text', 'date', 'upvotes', 'number of answers', 'shares',
       'views', 'environment', 'infrastructure', 'housing', 'economy',
       'life quality', 'aesthetics', 'government', 'technology',
       'environment sentiment', 'environment sentiment degree',
       'infrastructure sentiment', 'infrastructure sentiment degree',
       'housing sentiment', 'housing sentiment degree', 'economy sentiment',
       'economy sentiment degree', 'life quality sentiment',
       'life quality sentiment degree', 'aesthetics sentiment',
       'aesthetics sentiment degree', 'government sentiment',
       'government sentiment degree', 'technology sentiment',
       'technology sentiment degree'],
      dtype='object')


In [32]:
print(matched_posts)
print(len(posts))
print(more_than_one_theme_posts)

62
62
21


In [34]:
posts.to_json('quora_ABSA_entire_dataframe.json', orient='records', indent=4)

In [36]:
posts.head(30)

,ids,text,date,upvotes,number of answers,shares,views,environment,infrastructure,housing,...,economy sentiment,economy sentiment degree,life quality sentiment,life quality sentiment degree,aesthetics sentiment,aesthetics sentiment degree,government sentiment,government sentiment degree,technology sentiment,technology sentiment degree
0,QW5zd2VyQDA6MTA1ODI1Mzg4,Pre-Sales and Bid-Management have immense pote...,2018-10-28T18:50:07.259Z,87,3,2,35079,False,False,False,...,Positive,0.734731,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QW5zd2VyQDA6MTA2MzYxOTMw,"That is a big subject, depending on the scale ...",2018-11-01T16:59:46.862Z,0,2,1,250,False,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,QW5zd2VyQDA6MTA2MzYyMjMw,"Since about 2016, maybe 2015, I have been seei...",2018-11-01T17:02:28.168Z,0,3,0,209,False,False,False,...,Positive,0.540883,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,QW5zd2VyQDA6MTAwNzQzODk4,"I can’t answer for anybody else, but I can tel...",2018-09-23T03:38:25.067Z,14,8,0,2234,False,False,False,...,NaN,NaN,Negative,0.60119,NaN,NaN,NaN,NaN,NaN,NaN
4,QW5zd2VyQDA6MTEzNDQ5ODE1,TL;DR: It is possible for the US authorities t...,2018-12-20T14:52:12.725Z,0,255,0,452,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,Negative,0.716116,NaN,NaN
5,QW5zd2VyQDA6MTI4NTQwOTIy,It will probably be a part of gaming landscape...,2019-03-18T16:57:18.736Z,3,7,0,3391,False,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Negative,0.712674
6,QW5zd2VyQDA6MTM3NDE1Nzk2,"Back in June 2018, Apple hinted that it's comp...",2019-04-29T13:54:27.500Z,2,1,1,3506,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.920418
7,QW5zd2VyQDA6MTM4MDQxMTU=,"Yes, much. Iceland had and has a functional ...",2015-07-06T01:11:11.043Z,7,3,0,431,False,False,False,...,Negative,0.615299,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,QW5zd2VyQDA6MTMwMTcyODg=,"No, it's not. Actually it's a new beginning fo...",2015-06-04T16:24:29.838Z,0,2,1,307,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.642916
9,QW5zd2VyQDA6MTQ2MTg5Mzg1,It won’t. That’s the problem with “cloud gamin...,2019-06-08T03:25:43.483Z,0,1,0,239,False,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Negative,0.464609


In [37]:
print(posts.columns)

Index(['ids', 'text', 'date', 'upvotes', 'number of answers', 'shares',
       'views', 'environment', 'infrastructure', 'housing', 'economy',
       'life quality', 'aesthetics', 'government', 'technology',
       'environment sentiment', 'environment sentiment degree',
       'infrastructure sentiment', 'infrastructure sentiment degree',
       'housing sentiment', 'housing sentiment degree', 'economy sentiment',
       'economy sentiment degree', 'life quality sentiment',
       'life quality sentiment degree', 'aesthetics sentiment',
       'aesthetics sentiment degree', 'government sentiment',
       'government sentiment degree', 'technology sentiment',
       'technology sentiment degree'],
      dtype='object')


In [39]:
# calculating average sentiment based on theme:
# Weigh all posts by their degree in the numerator and denominator, means that the average sentiment will just be +/- 1 if there are only positive or negative themes but other than that does a pretty good job of weighing neutrality
def avg_sentiment_calculation(theme):
    theme_posts = posts[posts[theme] == True]
    if len(theme_posts) > 0:
        pos = theme_posts.loc[theme_posts[f"{theme} sentiment"] == "Positive", f"{theme} sentiment degree"].sum()
        neg = theme_posts.loc[theme_posts[f"{theme} sentiment"] == "Negative", f"{theme} sentiment degree"].sum()
        total = theme_posts[f"{theme} sentiment degree"].sum()
        return len(theme_posts), (pos-neg)/total
    else:
        return 0, None


print("Number of environmental posts: ", avg_sentiment_calculation("environment")[0], ", Average sentiment of environmental posts: ", avg_sentiment_calculation("environment")[1], sep="")
print("Number of infrastructural posts: ", avg_sentiment_calculation("infrastructure")[0], ", Average sentiment of infrastructural posts: ", avg_sentiment_calculation("infrastructure")[1], sep="")
print("Number of housing-related posts: ", avg_sentiment_calculation("housing")[0], ", Average sentiment of housing-related posts: ", avg_sentiment_calculation("housing")[1], sep="")
print("Number of economic posts: ", avg_sentiment_calculation("economy")[0], ", Average sentiment of economic posts: ", avg_sentiment_calculation("economy")[1], sep="")
print("Number of life-quality-related posts: ", avg_sentiment_calculation("life quality")[0], ", Average sentiment of life-quality-related posts: ", avg_sentiment_calculation("life quality")[1], sep="")
print("Number of aesthetics-related posts: ", avg_sentiment_calculation("aesthetics")[0], ", Average sentiment of aesthetics-related posts: ", avg_sentiment_calculation("aesthetics")[1], sep="")
print("Number of governmental posts: ", avg_sentiment_calculation("government")[0], ", Average sentiment of governmental posts: ", avg_sentiment_calculation("government")[1], sep="")
print("Number of technological posts: ", avg_sentiment_calculation("technology")[0], ", Average sentiment of technological posts: ", avg_sentiment_calculation("technology")[1], sep="")

Number of environmental posts: 1, Average sentiment of environmental posts: 1.0
Number of infrastructural posts: 25, Average sentiment of infrastructural posts: 0.029311440221548874
Number of housing-related posts: 0, Average sentiment of housing-related posts: None
Number of economic posts: 9, Average sentiment of economic posts: 0.5584811496739934
Number of life-quality-related posts: 1, Average sentiment of life-quality-related posts: -1.0
Number of aesthetics-related posts: 0, Average sentiment of aesthetics-related posts: None
Number of governmental posts: 4, Average sentiment of governmental posts: -0.8176488816850046
Number of technological posts: 43, Average sentiment of technological posts: 0.3429498205017289


In [17]:
posts['year'] = pd.to_datetime(posts['date']).dt.year

year_datasets = {year: posts[posts['year'] == year] for year in range(2010, 2027)}

posts_2010 = year_datasets[2010]
posts_2020 = year_datasets[2020]